In [6]:
# 1. DEPENDENCY INSTALLATION (Google Colab)
!pip install -q qiskit qiskit-aer qiskit-optimization qiskit-algorithms networkx pandas matplotlib pylatexenc qiskit-ibm-runtime
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2, FakeNairobiV2
from qiskit_optimization.applications import Maxcut
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
import warnings
warnings.filterwarnings('ignore')
dirs = {'tables': 'results/tables', 'figures': 'results/figures'}
for path in dirs.values():
    os.makedirs(path, exist_ok=True)
n = 5
G = nx.Graph()
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]
G.add_edges_from(edges)
plt.figure(figsize=(6, 4))
pos = nx.spring_layout(G, seed=42)
xn = nx
xn.draw(G, pos, with_labels=True, node_color='lightblue', node_size=800, font_weight='bold')
plt.savefig(f"{dirs['figures']}/maxcut_graph.png", dpi=300, bbox_inches='tight')
plt.close()
maxcut = Maxcut(G)
qp = maxcut.to_quadratic_program()
observable, offset = qp.to_ising()
processors = {'Processor_A': AerSimulator.from_backend(FakeManilaV2()), 'Processor_B': AerSimulator.from_backend(FakeNairobiV2())}
def benchmark_qaoa(processor_name, backend, observable):
    # Bind the Sampler primitive to the specific fake backend
sampler = BackendSampler(backend=backend, options={'shots': 1024})
    optimizer = COBYLA(maxiter=50)
    qaoa = QAOA(sampler, optimizer, reps=2)
    start_time = time.time()
    result = qaoa.compute_minimum_eigenvalue(observable)
    execution_time = time.time() - start_time
    best_bitstring = None
    max_prob = 0
    probabilities = {}
    for i, state_result in enumerate(result.eigenstate):
        bitstring = format(i, f'0{observable.num_qubits}b')
        prob = np.abs(state_result)**2
        probabilities[bitstring] = prob
        if prob > max_prob:
            max_prob = prob
            best_bitstring = bitstring
    x = [int(bit) for bit in best_bitstring][::-1]
    cut_value = maxcut.objective_value(x)
    return {'Processor': processor_name, 'Execution_Time_sec': round(execution_time, 2), 'Max_Cut_Value': cut_value}
results_data = []
for name, backend in processors.items():
    results_data.append(benchmark_qaoa(name, backend, observable))
df_results = pd.DataFrame(results_data)
csv_path = f"{dirs['tables']}/PS_O1_benchmark_results.csv"
df_results.to_csv(csv_path, index=False)
print(df_results)

IndentationError: expected an indented block after function definition on line 34 (4276961500.py, line 36)